In [1]:


import sys
sys.path.append("../../")

In [2]:
from verbatim_rag.index import VerbatimIndex
from verbatim_rag.vector_stores import LocalMilvusStore
from verbatim_rag.embedding_providers import SentenceTransformersProvider

### Laden der Datenbank

In [3]:

db_path = "../../data/clapnq_milvus_e5_v4.db"

dense_provider = SentenceTransformersProvider(
    model_name="intfloat/e5-base-v2",
    device="cpu"
)

# Verbinde mit existierender DB
vector_store = LocalMilvusStore(
    db_path=db_path,
    collection_name="clapnq_passages_e5",
    enable_dense=True,
    enable_sparse=False,
    dense_dim=768,
    milvus_connection_timeout=300,
)

# Index erstellen (lädt existierende Daten)
index = VerbatimIndex(
    vector_store=vector_store,
    dense_provider=dense_provider,
    chunker_provider=None,
)

/home/pkunerth/projects/verbatim-rag/.venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


### Laden der Dev Daten

In [4]:
import json

def load_jsonl(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

dev_answerable = load_jsonl('../../data/clapnq_dev_answerable.jsonl')
dev_unanswerable = load_jsonl('../../data/clapnq_dev_unanswerable.jsonl')

print(f"Answerable: {len(dev_answerable)}")
print(f"Unanswerable: {len(dev_unanswerable)}")

print(f"\nExample:")
print(f"Question: {dev_answerable[0]['input']}")
print(f"Gold passage title: {dev_answerable[0]['passages'][0]['title']}")
print(f"Gold passage text: {dev_answerable[0]['passages'][0]['text'][:200]}...")

Answerable: 300
Unanswerable: 300

Example:
Question: which method of forecasting uses averages to predict future weather
Gold passage title: Forecasting
Gold passage text: Seasonality is a characteristic of a time series in which the data experiences regular and predictable changes which recur every calendar year . Any predictable change or pattern in a time series that...


In [5]:
# Teste mit einem Beispiel
example = dev_answerable[0]

question = example['input']
gold_passage_text = example['passages'][0]['text']

print(f"Question: {question}")
print(f"Gold passage (first 100 chars): {gold_passage_text[:100]}...")

# Retrieve Top-10
results = index.query(text="query: " + question, k=20)

# Vergleiche
print(f"\nTop 10 Retrieved - Is gold passage there?")
found = False
for i, result in enumerate(results, 1):
    is_gold = result.text.strip().removeprefix("passage: ").strip() == gold_passage_text.strip()
    print(f"  {i}. {'✅ GOLD!' if is_gold else '❌'} {result.metadata['title']} (score: {result.score:.4f})")
    if is_gold:
        found = True

print(f"\nGold passage found in Top-10: {'✅ YES' if found else '❌ NO'}")

Question: which method of forecasting uses averages to predict future weather
Gold passage (first 100 chars): Seasonality is a characteristic of a time series in which the data experiences regular and predictab...

Top 10 Retrieved - Is gold passage there?
  1. ❌ Forecasting (score: 0.8596)
  2. ❌ Forecasting (score: 0.8579)
  3. ❌ Forecasting (score: 0.8530)
  4. ❌ Forecasting (score: 0.8499)
  5. ❌ Forecasting (score: 0.8473)
  6. ❌ Forecasting (score: 0.8464)
  7. ❌ Forecasting (score: 0.8455)
  8. ❌ Forecasting (score: 0.8422)
  9. ❌ Forecasting (score: 0.8419)
  10. ❌ Forecasting (score: 0.8417)
  11. ❌ Forecasting (score: 0.8417)
  12. ❌ Forecasting (score: 0.8410)
  13. ❌ Forecasting (score: 0.8406)
  14. ❌ Forecasting (score: 0.8401)
  15. ❌ Forecasting (score: 0.8398)
  16. ❌ Forecasting (score: 0.8376)
  17. ❌ Forecasting (score: 0.8367)
  18. ❌ Forecasting (score: 0.8357)
  19. ❌ Forecasting (score: 0.8344)
  20. ❌ Forecasting (score: 0.8342)

Gold passage found in Top-10: ❌

In [6]:
from difflib import SequenceMatcher
import numpy as np

def is_gold_match(retrieved_text, gold_text, threshold=0.9):
    a = retrieved_text.strip().removeprefix("passage: ").strip()
    b = gold_text.strip()
    if b in a:
        return True
    return SequenceMatcher(None, a, b).ratio() > threshold

def evaluate_retrieval(data, index, k=10, n_samples=None):
    
    if n_samples:
        data = data[:n_samples]
    
    recall_list = []
    ndcg_1_list = []
    ndcg_3_list = []
    ndcg_5_list = []
    ndcg_10_list = []
    
    for i, example in enumerate(data):
        question = example['input']
        gold_text = example['passages'][0]['text'].strip()
        
        results = index.query(text="query: " + question, k=k)
        
        gold_position = None
        for j, result in enumerate(results):
            if is_gold_match(result.text, gold_text):
                gold_position = j + 1
                break
        
        recall_list.append(1.0 if gold_position else 0.0)
        
        if gold_position and gold_position <= 1:
            ndcg_1_list.append(1.0 / np.log2(gold_position + 1))
        else:
            ndcg_1_list.append(0.0)
        
        if gold_position and gold_position <= 3:
            ndcg_3_list.append(1.0 / np.log2(gold_position + 1))
        else:
            ndcg_3_list.append(0.0)
        
        if gold_position and gold_position <= 5:
            ndcg_5_list.append(1.0 / np.log2(gold_position + 1))
        else:
            ndcg_5_list.append(0.0)
        
        if gold_position and gold_position <= 10:
            ndcg_10_list.append(1.0 / np.log2(gold_position + 1))
        else:
            ndcg_10_list.append(0.0)
    
    print(f"\n{'='*50}")
    print(f"RESULTS (n={len(data)}, k={k})")
    print(f"{'='*50}")
    print(f"nDCG@1:    {np.mean(ndcg_1_list):.4f}")
    print(f"nDCG@3:    {np.mean(ndcg_3_list):.4f}")
    print(f"nDCG@5:    {np.mean(ndcg_5_list):.4f}")
    print(f"nDCG@10:   {np.mean(ndcg_10_list):.4f}")
    print(f"Recall@10: {np.mean(recall_list):.4f}")
    print(f"{'='*50}")

evaluate_retrieval(dev_answerable, index, k=10, n_samples=10)


RESULTS (n=10, k=10)
nDCG@1:    0.6000
nDCG@3:    0.7262
nDCG@5:    0.7262
nDCG@10:   0.7551
Recall@10: 0.9000


In [7]:
# Jetzt für alle 300 answerable questions
evaluate_retrieval(dev_answerable, index, k=10, n_samples=None)


RESULTS (n=300, k=10)
nDCG@1:    0.4433
nDCG@3:    0.6010
nDCG@5:    0.6372
nDCG@10:   0.6643
Recall@10: 0.8833


In [8]:

from tabulate import tabulate
 
# ── Paper baselines (Dev Set, Table 4 aus Rosenthal et al. 2025) ──
paper_baselines_retrieval = [
    ("BM25",              18,   30,   35,   40,   67),
    ("all-MiniLM-L6-v2", 29,   43,   48,   53,   79),
    ("BGE-base",          37,   54,   59,   61,   85),
    ("E5-base-v2",        41,   57,   61,   64,   87),
]
 
# ── Unsere Ergebnisse (Dev Set, n=300) ──
our_results_retrieval = [
    ("VerbatimRAG (E5-base-v2)", 44.3, 60.1, 63.7, 66.4, 88.3),
]
 
headers_retrieval = [
    "Model", "nDCG@1", "nDCG@3", "nDCG@5", "nDCG@10", "R@10"
]
 
all_rows = list(paper_baselines_retrieval) + list(our_results_retrieval)
 
# Markdown (für README)
print("=== MARKDOWN ===")
print(tabulate(all_rows, headers=headers_retrieval, tablefmt="pipe",
               floatfmt=".1f"))
 
# LaTeX booktabs (für Thesis)
print("\n=== LATEX ===")
latex = tabulate(all_rows, headers=headers_retrieval, tablefmt="latex_booktabs",
                 floatfmt=".1f")
 
# Trennlinie vor unseren Ergebnissen einfügen
n_paper = len(paper_baselines_retrieval)
lines = latex.split("\n")
# Finde die Zeile nach den Paper-Baselines und füge \midrule ein
row_lines = [l for l in lines if l.startswith(" ") or l.startswith("B") or
             l.startswith("a") or l.startswith("E") or l.startswith("V")]
# Einfacher Approach: midrule nach n_paper Datenzeilen einfügen
data_count = 0
result_lines = []
for line in lines:
    result_lines.append(line)
    # Zähle Datenzeilen (enthalten & als Trennzeichen in booktabs)
    if "&" in line:
        data_count += 1
        if data_count == n_paper:
            result_lines.append(r"\midrule")
print("\n".join(result_lines))

=== MARKDOWN ===
| Model                    |   nDCG@1 |   nDCG@3 |   nDCG@5 |   nDCG@10 |   R@10 |
|:-------------------------|---------:|---------:|---------:|----------:|-------:|
| BM25                     |     18.0 |     30.0 |     35.0 |      40.0 |   67.0 |
| all-MiniLM-L6-v2         |     29.0 |     43.0 |     48.0 |      53.0 |   79.0 |
| BGE-base                 |     37.0 |     54.0 |     59.0 |      61.0 |   85.0 |
| E5-base-v2               |     41.0 |     57.0 |     61.0 |      64.0 |   87.0 |
| VerbatimRAG (E5-base-v2) |     44.3 |     60.1 |     63.7 |      66.4 |   88.3 |

=== LATEX ===
\begin{tabular}{lrrrrr}
\toprule
 Model                    &   nDCG@1 &   nDCG@3 &   nDCG@5 &   nDCG@10 &   R@10 \\
\midrule
 BM25                     &     18.0 &     30.0 &     35.0 &      40.0 &   67.0 \\
 all-MiniLM-L6-v2         &     29.0 &     43.0 &     48.0 &      53.0 &   79.0 \\
 BGE-base                 &     37.0 &     54.0 &     59.0 &      61.0 &   85.0 \\
\midrule
 E5-